In [16]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.output_parsers import JsonOutputParser
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator
import os

In [17]:
load_dotenv()

True

In [18]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    temperature=0.7,
    max_new_tokens=500
)

model = ChatHuggingFace(llm=llm)

In [19]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description='score out of 10', ge=0, le=10)

In [20]:
parser = JsonOutputParser(pydantic_object=EvaluationSchema)

def invoke_structured(prompt_text: str) -> EvaluationSchema:
    """Helper to invoke model and parse structured output."""
    format_instructions = parser.get_format_instructions()
    full_prompt = f"{prompt_text}\n\nRespond ONLY with valid JSON (no extra text).\n{format_instructions}"
    raw = model.invoke(full_prompt)
    parsed = parser.parse(raw.content)
    return EvaluationSchema(**parsed)

In [21]:
essay = """  Importance of Technology in Modern Life

Technology has become an essential part of modern life. It has transformed the way people communicate, learn, work, and access information. With the help of smartphones and the internet, people can connect with others across the world within seconds. Technology has also improved healthcare through advanced medical equipment and digital health records.

In education, online learning platforms provide students with access to knowledge from anywhere. Businesses use technology to increase productivity and make better decisions through data analysis. Furthermore, technology has made daily tasks more convenient, saving both time and effort.

However, excessive dependence on technology can lead to challenges such as reduced physical activity, privacy concerns, and cyber threats. Therefore, it is important to use technology responsibly.

In conclusion, technology plays a vital role in improving the quality of life. When used wisely, it can contribute significantly to individual growth and societal development.
 """

In [ ]:
# This cell is for openAI models to get a structured output

# structured_model = model.with_structured_output(EvaluationSchema)

# prompt = f'Evaluate the quality of the following essay and provide a feedbak and assign a score out od 10 \n {essay}'
# structured_model.invoke(prompt)

In [23]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [24]:
def evaluate_language(state: UPSCState):

    prompt = f'Evaluate the LANGUAGE QUALITY of the following essay and provide feedback and assign a score out of 10 \n {state["essay"]}'
    output = invoke_structured(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [25]:
def evaluate_analysis(state: UPSCState):

    prompt = f'Evaluate the DEPTH OF ANALYSIS of the following essay and provide feedback and assign a score out of 10 \n {state["essay"]}'
    output = invoke_structured(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [26]:
def evaluate_thought(state: UPSCState):

    prompt = f'Evaluate the CLARITY OF THOUGHT of the following essay and provide feedback and assign a score out of 10 \n {state["essay"]}'
    output = invoke_structured(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [27]:
def final_evaluation(state: UPSCState):

    prompt = (
        f"Based on the following feedbacks, create a summarized overall feedback:\n"
        f"Language Quality: {state['language_feedback']}\n"
        f"Depth of Analysis: {state['analysis_feedback']}\n"
        f"Clarity of Thought: {state['clarity_feedback']}"
    )
    overall_feedback = model.invoke(prompt)

    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])
    return {'overall_feedback': overall_feedback.content, 'avg_score': avg_score}

In [28]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()
print('Graph compiled successfully')

Graph compiled successfully


In [29]:
initial_state = {
    'essay': essay,
    'individual_scores': []
}

result = workflow.invoke(initial_state)
print(f"Average Score: {result['avg_score']:.1f}/10")
print(f"\nOverall Feedback:\n{result['overall_feedback']}")

Average Score: 7.7/10

Overall Feedback:
Overall Feedback:
The essay effectively highlights the importance of technology in modern life, covering various aspects such as communication, education, and healthcare. It also discusses the potential downsides of excessive technology reliance. However, to enhance its quality, the essay could benefit from more detailed analysis, specific examples, and a deeper discussion of the impact on different demographics or societies. The structure and clarity of the essay are strong, but adding more concrete examples and a bit more depth in both the benefits and drawbacks would strengthen the argument.


In [30]:
initial_state = {
    'essay': essay
}

workflow.invoke(initial_state)

{'essay': '  Importance of Technology in Modern Life\n\nTechnology has become an essential part of modern life. It has transformed the way people communicate, learn, work, and access information. With the help of smartphones and the internet, people can connect with others across the world within seconds. Technology has also improved healthcare through advanced medical equipment and digital health records.\n\nIn education, online learning platforms provide students with access to knowledge from anywhere. Businesses use technology to increase productivity and make better decisions through data analysis. Furthermore, technology has made daily tasks more convenient, saving both time and effort.\n\nHowever, excessive dependence on technology can lead to challenges such as reduced physical activity, privacy concerns, and cyber threats. Therefore, it is important to use technology responsibly.\n\nIn conclusion, technology plays a vital role in improving the quality of life. When used wisely,